# **Day 77: Web Scraping with BeautifulSoup**

**Web scraping** means reading a web page with a program and **taking out the data** we need.

A web page is written in **HTML**. HTML uses **tags** like `<h1>` (heading), `<p>` (paragraph), `<table>` (table) and `<a>` (link).

We use the `beautifulsoup4` library (`pip install beautifulsoup4`).

**Rules for scraping politely:**
1. Check the website's rules (its `robots.txt` file and terms of use).
2. Do not send too many requests quickly.
3. If the website gives an API, use the API instead.

For practice, we will use a small HTML page that we write ourselves.

In [1]:
html = """
<html>
<head><title>Indian Earth Observation Satellites</title></head>
<body>
  <h1 id="top">Indian Remote Sensing Satellites</h1>
  <p class="intro">A small list for practice. Details are simplified.</p>

  <div class="satellite" data-agency="ISRO">
    <h2 class="name">Resourcesat-2A</h2>
    <span class="launch">2016</span>
    <ul class="sensors">
      <li data-res="5.8">LISS-IV</li>
      <li data-res="23.5">LISS-III</li>
      <li data-res="56">AWiFS</li>
    </ul>
    <a href="/sat/resourcesat-2a">details</a>
  </div>

  <div class="satellite" data-agency="ISRO">
    <h2 class="name">Cartosat-2</h2>
    <span class="launch">2007</span>
    <ul class="sensors">
      <li data-res="0.8">PAN</li>
    </ul>
    <a href="/sat/cartosat-2">details</a>
  </div>

  <div class="satellite featured" data-agency="ISRO-NASA">
    <h2 class="name">NISAR</h2>
    <span class="launch">2025</span>
    <ul class="sensors">
      <li data-res="10">L-band SAR</li>
      <li data-res="10">S-band SAR</li>
    </ul>
    <a href="https://example.com/nisar">details</a>
  </div>

  <table id="uses">
    <tr><th>Application</th><th>Satellite data used</th></tr>
    <tr><td>Crop monitoring</td><td>Resourcesat</td></tr>
    <tr><td>Urban mapping</td><td>Cartosat</td></tr>
    <tr><td>Flood mapping</td><td>SAR</td></tr>
  </table>
</body>
</html>
"""

In [2]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(html, "html.parser")
print(soup.title.string)
print(soup.h1.text)
print(soup.find("p", class_="intro").get_text())

Indian Earth Observation Satellites
Indian Remote Sensing Satellites
A small list for practice. Details are simplified.


## **find() and find_all()**
- `find()` gives the **first** matching tag.
- `find_all()` gives **all** matching tags as a list.

In [3]:
for h2 in soup.find_all("h2", class_="name"):
    print(h2.get_text(strip=True))

Resourcesat-2A
Cartosat-2
NISAR


## **CSS Selectors: select() and select_one()**
This is the easiest way. The patterns work like CSS: `#id` for an id, `.class` for a class, and a space for "inside".

In [4]:
print(soup.select_one("#top").text)
print([li.text for li in soup.select("div.satellite ul.sensors li")])
print([div["data-agency"] for div in soup.select("div.satellite")])
print(soup.select_one("div.featured h2").text)

Indian Remote Sensing Satellites
['LISS-IV', 'LISS-III', 'AWiFS', 'PAN', 'L-band SAR', 'S-band SAR']
['ISRO', 'ISRO', 'ISRO-NASA']
NISAR


## **Taking Out Structured Data**

In [5]:
satellites = []
for div in soup.select("div.satellite"):
    satellites.append({
        "name": div.select_one(".name").get_text(strip=True),
        "launch": int(div.select_one(".launch").text),
        "agency": div["data-agency"],
        "sensors": {li.text: float(li["data-res"]) for li in div.select("li")},
        "link": div.a["href"],
    })

for s in satellites:
    best = min(s["sensors"].items(), key=lambda kv: kv[1])
    print(f"{s['name']:<15} launched {s['launch']}  best sensor: {best[0]} ({best[1]} m)")

Resourcesat-2A  launched 2016  best sensor: LISS-IV (5.8 m)
Cartosat-2      launched 2007  best sensor: PAN (0.8 m)
NISAR           launched 2025  best sensor: L-band SAR (10.0 m)


Let us understand the code above:
- `div.select_one(".name")` searches **only inside** that one `div`.
- `li["data-res"]` reads an **attribute** of the tag.
- `div.a["href"]` gets the link address of the first `<a>` tag inside the div.

## **Reading a Table**

In [6]:
rows = soup.select("#uses tr")
header = [th.text for th in rows[0].find_all("th")]
table = [dict(zip(header, [td.text for td in row.find_all("td")])) for row in rows[1:]]
for t in table:
    print(t)

{'Application': 'Crop monitoring', 'Satellite data used': 'Resourcesat'}
{'Application': 'Urban mapping', 'Satellite data used': 'Cartosat'}
{'Application': 'Flood mapping', 'Satellite data used': 'SAR'}


## **Making Links Complete**
Some links are **relative** (like `/sat/cartosat-2`). `urljoin` makes them complete.

In [7]:
from urllib.parse import urljoin
base = "https://satellite-list.example/"
for a in soup.find_all("a"):
    print(urljoin(base, a["href"]))

https://satellite-list.example/sat/resourcesat-2a
https://satellite-list.example/sat/cartosat-2
https://example.com/nisar


## **Scraping a Real Page**
`example.com` is a website made for testing, so it is safe to use.

In [8]:
import requests
try:
    r = requests.get("https://example.com", timeout=10, headers={"User-Agent": "learning-python"})
    page = BeautifulSoup(r.text, "html.parser")
    print(page.title.text)
    print(page.p.get_text(strip=True)[:100])
except requests.RequestException as e:
    print("Could not connect:", e)

Example Domain
This domain is for use in documentation examples without needing permission. Avoid use in operations


## **When Scraping Becomes Hard**
- Pages made by **JavaScript** (the data is not in the HTML). Tools like Selenium or Playwright are needed.
- The website **changes its design**, and the scraper breaks.
- **Login** pages and **captchas**.

In [9]:
# Quick question
# What is the difference between soup.find("li") and soup.find_all("li")?